In [3]:
import torch, triton, triton.language as tl

@triton.jit
def copy_kernel(x_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    # TODO 1: offsets = the BLOCK element indices this program owns
    offsets = pid * BLOCK + tl.arange(0, BLOCK)
    # TODO 2: mask = which of those offsets are < n
    mask = offsets < n
    # TODO 3: load x at those offsets (use the mask)
    x = tl.load(x_ptr + offsets, mask=mask)
    # TODO 4: store it to out_ptr at the same offsets (use the mask)
    tl.store(out_ptr + offsets, x, mask=mask)

def copy(x):
    out = torch.empty_like(x)
    n = x.numel()
    # TODO 5: grid = number of programs needed to cover n elements
    BLOCK = 1024
    grid = (triton.cdiv(n, BLOCK),)
    copy_kernel[grid](x, out, n, BLOCK=BLOCK)
    return out

In [4]:
import torch, triton, triton.language as tl


@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    # TODO 1: offsets = the BLOCK element indices this program owns
    offsets = pid * BLOCK + tl.arange(0, BLOCK)
    # TODO 2: mask = which of those offsets are < n
    mask = offsets < n
    # TODO 3: load x and y at those offsets (use the mask)
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    # TODO 4: compute x + y
    z = x + y
    # TODO 5: store the result to out_ptr at the same offsets (use the mask)
    tl.store(out_ptr + offsets, z, mask=mask)


def add(x, y):
    out = torch.empty_like(x)
    n = x.numel()
    BLOCK = 1024
    # TODO 6: grid = number of programs needed to cover n elements
    grid = (triton.cdiv(n, BLOCK),)
    add_kernel[grid](x, y, out, n, BLOCK=BLOCK)
    return out

In [7]:
import torch, triton, triton.language as tl


@triton.jit
def strided_copy_kernel(x_ptr, out_ptr, n_cols, x_row_stride, out_row_stride,
                        BLOCK: tl.constexpr):
    """
    Copy one row of a 2-D tensor per program.

    Program `pid = tl.program_id(0)` owns row `pid`. Within a row the elements are
    contiguous (column stride 1), but consecutive rows are `x_row_stride` elements
    apart in `x` and `out_row_stride` elements apart in `out`. Those two strides
    can differ from each other and from `n_cols`.

    BLOCK is a power of 2 >= n_cols, so one program covers the whole row.
    Lanes with column index >= n_cols must neither read nor write memory.
    The address of element (row, col) is: base_ptr + row * row_stride + col.
    """
    pid = tl.program_id(0)

    column_index = tl.arange(0, BLOCK)
  
    in_offsets = pid * x_row_stride + column_index
    out_offsets = pid * out_row_stride + column_index
    mask = column_index < n_cols

    x = tl.load(x_ptr + in_offsets, mask=mask)

    tl.store(out_ptr + out_offsets, x, mask=mask)


def strided_copy(x: torch.Tensor) -> torch.Tensor:
    """
    Return a new contiguous (n_rows, n_cols) tensor equal to `x`.

    `x` is 2-D with stride(1) == 1 but any stride(0), so a view of a wider tensor
    is a valid input. Launch one program per row. BLOCK is the smallest power of 2
    that is >= n_cols (use triton.next_power_of_2).

    Cases it must handle, with n_cols = 100 in the first two:
      * x = big[:, :100] where big is (64, 256): stride(0) = 256, stride(1) = 1
      * x = big[::2, :100]: stride(0) = 512, so rows are skipped
      * a contiguous x, where stride(0) == n_cols
      * n_cols = 1 and n_cols = 1024
    """
    a, b = x.shape
    out = torch.empty_like(x)
    BLOCK = triton.next_power_of_2(b)
    grid = (a,)
    strided_copy_kernel[grid](x, out, b, x.stride(0), out.stride(0), BLOCK=BLOCK)
    return out

In [ ]:
import torch, triton, triton.language as tl


@triton.jit
def softmax_kernel(x_ptr, out_ptr, n_cols, x_row_stride, out_row_stride,
                   BLOCK: tl.constexpr):
    """Row-wise softmax, one row per program. Must be numerically stable."""
    pid = tl.program_id(0)

    column_index = tl.arange(0, BLOCK)
    in_offsets = pid * x_row_stride + column_index
    out_offsets = pid * out_row_stride + column_index
    mask = column_index < n_cols

    x = tl.load(x_ptr + in_offsets, mask=mask, other=float("-inf"))
    x32 = x.to(tl.float32)
    x32 -= x32.max()
    x32 = tl.exp(x32)
    x32 = x32 / x32.sum()

    tl.store(out_ptr + out_offsets, x32, mask=mask)


def softmax(x: torch.Tensor) -> torch.Tensor:
    """
    Softmax along the last dim of a GPU tensor of shape [B1, ..., T, d]
    (any number of leading dims) with stride(-1) == 1.
    Returns a tensor of the same shape and dtype as x.
    Must match torch.softmax(x.float(), dim=-1).to(x.dtype).
    """
    d = x.shape[-1]
    out = torch.empty(x.shape, dtype=x.dtype, device=x.device)
    x2d = x.reshape(-1, d)
    out2d = out.reshape(-1, d)
    BLOCK = triton.next_power_of_2(d)
    grid = (x2d.shape[0], )
    softmax_kernel[grid](x2d, out2d, d, x2d.stride(0), out2d.stride(0), BLOCK)
    return out


In [ ]:
import torch, triton, triton.language as tl


@triton.jit
def rmsnorm_fwd_kernel(x_ptr, gamma_ptr, out_ptr, rstd_ptr, n_cols,
                       x_row_stride, out_row_stride, eps,
                       BLOCK: tl.constexpr):
    """RMSNorm forward, one row per program. Also stores each row's rstd (fp32)."""

    pid = tl.program_id(0)
    column_index = tl.arange(0, BLOCK)
    x_offsets = pid * x_row_stride + column_index
    out_offsets = pid * out_row_stride + column_index
    mask = column_index < n_cols

    x = tl.load(x_ptr + x_offsets, mask=mask, other=0)
    x = x.to(tl.float32)
    rstd = 1 / tl.sqrt((x * x).sum() / n_cols + eps)
    tl.store(rstd_ptr + pid, rstd)
    gamma = tl.load(gamma_ptr + column_index, mask=mask)
    tl.store(out_ptr + out_offsets, gamma * x * rstd, mask=mask)
    


@triton.jit
def rmsnorm_bwd_kernel(grad_out_ptr, x_ptr, gamma_ptr, rstd_ptr,
                       grad_x_ptr, grad_gamma_partial_ptr, n_rows, n_cols,
                       grad_out_row_stride, x_row_stride, grad_x_row_stride,
                       ROWS_PER_PROGRAM: tl.constexpr, BLOCK: tl.constexpr):
    """
    RMSNorm backward. Each program handles ROWS_PER_PROGRAM rows, writes their grad_x,
    and writes one fp32 partial sum of grad_gamma, so no (n_rows, d) buffer exists.
    """

    pid = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols

    gamma = tl.load(gamma_ptr + cols, mask=mask, other=0).to(tl.float32)
    acc = tl.zeros([BLOCK], dtype=tl.float32)

    for i in range(ROWS_PER_PROGRAM):
        r = pid * ROWS_PER_PROGRAM + i
        rstd = tl.load(rstd_ptr + r)
        if r < n_rows:
            grad_out_offsets = r * grad_out_row_stride + cols
            x_offsets = r * x_row_stride + cols
            grad_x_offsets = r * grad_x_row_stride + cols
            grad_out = tl.load(grad_out_ptr + grad_out_offsets, mask=mask, other=0).to(tl.float32)
            x = tl.load(x_ptr + x_offsets, mask=mask, other=0).to(tl.float32)
            u = gamma * grad_out
            grad_x = rstd * u - (rstd * rstd * rstd) / n_cols * (u * x).sum() * x
            tl.store(grad_x_ptr + grad_x_offsets, grad_x, mask=mask)
            acc += grad_out * x * rstd
    
    tl.store(grad_gamma_partial_ptr + pid * n_cols + cols, acc, mask=mask)

            


    